### Step0: Install Environment
conda create --name py310 -y python=3.10.15

pip install -r requirement.txt

### Step1: Download OSM data in New York
In this study, we focus on "Buildings", "Landuse", "POI"

You can download from here: https://download.geofabrik.de/

We have provided the required data:

### Step2: Extract features within the study region (NYC)

python 1_1_extract_osm_features_in_region.py
    --region_shp "data/nyc_regions/region.shp"
    --osm_feature_shp "data/gis_osm_pois_free_1/gis_osm_pois_free_1.shp"
    --output_shp "data/nyc_osm_pois/pois.shp"
    --output_csv "data/nyc_osm_pois/pois.csv"

python 1_1_extract_osm_features_in_region.py 
    --region_shp "data/nyc_regions/region.shp" 
    --osm_feature_shp "data/gis_osm_buildings_a_free_1/gis_osm_buildings_a_free_1.shp" 
    --output_shp "data/nyc_osm_buildings/buildings.shp"
    --output_csv "data/nyc_osm_buildings/buildings.csv"

python 1_1_extract_osm_features_in_region.py 
    --region_shp "data/nyc_regions/region.shp" 
    --osm_feature_shp "data/gis_osm_landuse_a_free_1/gis_osm_landuse_a_free_1.shp" 
    --output_shp "data/nyc_osm_landuse/landuse.shp"
    --output_csv "data/nyc_osm_landuse/landuse.csv"
    

### Step3: Rasterize the features with h3

python 1_2_rasterization.py 
    --input_csv_path data/nyc_osm_buildings/buildings.csv 
    --output_csv_path data/nyc_buildings_h3.csv

python 1_2_rasterization.py 
    --input_csv_path data/nyc_osm_buildings/buildings.csv 
    --output_csv_path data/nyc_buildings_h3.csv


In [2]:
#### merge aoi files

import os
import pandas as pd
from typing import List, Optional

def concat_csv_files(
    file_paths: List[str],
    output_file: str,
    columns: Optional[List[str]] = None,
    ignore_index: bool = True,
    add_source: bool = False,
    encoding: Optional[str] = None
) -> pd.DataFrame:
    """
    Concatenate multiple CSV files, optionally keeping only selected columns,
    and save the result to an output CSV file.

    Args:
        file_paths: List of input CSV file paths.
        output_file: Path to the output CSV file.
        columns: List of columns to keep. If None, keep all columns.
        ignore_index: Whether to reset the index in the concatenated result.
        add_source: Whether to add a column with the source file name.
        encoding: Optional file encoding for reading/writing CSVs.

    Returns:
        The concatenated DataFrame.
    """
    dfs = []

    for fp in file_paths:
        if not os.path.exists(fp):
            print(f"Warning: {fp} not found, skipping.")
            continue

        df = pd.read_csv(fp, encoding=encoding)

        if columns is not None:
            missing_cols = [col for col in columns if col not in df.columns]
            if missing_cols:
                print(f"Warning: {fp} is missing columns {missing_cols}, skipping.")
                continue
            df = df[columns]

        if add_source:
            df["source_file"] = os.path.basename(fp)

        dfs.append(df)

    if not dfs:
        raise ValueError("No valid CSV files found.")

    result = pd.concat(dfs, ignore_index=ignore_index)

    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    result.to_csv(output_file, index=False, encoding=encoding)

    return result


files = ["data/nyc_buildings_h3.csv", "data/nyc_landuse_h3.csv"]

data = concat_csv_files(
    file_paths=files,
    output_file="data/aoi.csv",
    columns=["osm_id", "agg_category", "geometry", "h3_list", "poi_aoi"],
    add_source=False
)

,osm_id,agg_category,geometry,h3_list,poi_aoi
0,31974497,residential,POINT (-73.9673843946306 40.67906738559073),8b2a100da205fff,aoi
1,34633854,office:empire state building,POINT (-73.9850578581105 40.74843122871664),8b2a100d2d49fff,aoi
2,34633854,office:empire state building,POINT (-73.98598888616235 40.74878374676755),8b2a100d2d4afff,aoi
3,34633854,office:empire state building,POINT (-73.98534848824615 40.748045690473624),8b2a100d2896fff,aoi
4,34633854,office:empire state building,POINT (-73.985668684648 40.74841471816488),8b2a100d2d4bfff,aoi
...,...,...,...,...,...
72575,1137610816,retail,POINT (-73.76218265152225 40.74782058110505),8b2a100504d1fff,aoi
72576,1137610818,forest,POINT (-73.75784786501667 40.75510402345344),8b2a1005626bfff,aoi
72577,1137610818,forest,POINT (-73.75752947378393 40.75473437010585),8b2a10050536fff,aoi
72578,1137610818,forest,POINT (-73.75816626134745 40.75547367773417),8b2a1005626afff,aoi


### Step4: Generate SpaBert Input

python 1_3_generate_spabert_json.py 
    --csv_file_path data/nyc_osm_pois/pois.csv 
    --processed_aoi_csv_file_path data/aoi.csv


### Step5: Train SpaBERT and Generate SpaBERT Embedding

```
python 2_1_train_predict_spabert.py \
    --json_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100.json' \
    --model_save_dir='model_weights/' \
    --mode='train'
```

```
python 2_1_train_predict_spabert.py --json_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100.json' \
    --model_save_dir='model_weights/' \
    --csv_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_spabert_embedding.csv' \
    --mode='predict'
```


### Step6: Group SpaBERT embedding to form Region embedding 
```
python 2_2_region_embedding.py \
    --in_csv_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_spabert_embedding.csv' \
    --out_csv_file_path='data/nyc_osm_pois/pois/pois_pseudo_sentence_v2_nn100_sdm100_region_embedding.csv'
```